# PaveScan AI v2 — Crack Detection Model Training

**Train a YOLO11 model to detect pavement cracks and road damage.**

This notebook is designed to run in **Google Colab with a GPU runtime**.

## What's new in v2.1 (and v2.2 additions):
- **YOLO11** architecture (was YOLOv8n) — much more accurate
- **1280px** training resolution (was 640px) — 4x more pixel detail
- **Auto GPU tier detection** — picks YOLO11l on A100, YOLO11m on T4
- **Aggressive augmentation** — mosaic, mixup, copy-paste, rotation, scale, shear, perspective, hsv jitter
- **Multi-scale training** — varies imgsz during training for robustness
- **200 epochs / patience=50** — longer training to squeeze out final mAP
- **v1-vs-v2 comparison** — evaluates both models on the same val set, saves a `metrics_v2_*.json` you can drop into your README
- **Cross-domain validation (NEW, Option B)** — trains on 5 countries, evaluates on held-out Japan as a true generalization test, saves `metrics_v2_japan_holdout.json`
- **Confidence calibration (NEW)** — fits isotonic regression on val set so raw confidence scores actually mean P(true positive); saves `calibration_v2_seg.json` / `calibration_v2_det.json`
- **Run-All friendly** — pick one option, click Run All, walk away

## How to use:
1. Open this notebook in **Google Colab**
2. Go to **Runtime → Change runtime type → A100 (Colab Pro recommended)** or T4 GPU
3. **(Optional but recommended)** Upload your existing v1 weights so the notebook can run a v1-vs-v2 comparison:
   - Click the folder icon on the left sidebar in Colab
   - Drag `pavescan_crack_seg.pt` (or `pavescan_rdd2022.pt`) into `/content/`
   - The comparison cell will auto-detect them
4. In the "Pick ONE Training Option" cell below, set `TRAINING_OPTION` to `"A"`, `"B"`, or `"C"`
5. Click **Runtime → Run All**
6. Wait. Keep the tab open.
7. The trained `.pt` file PLUS metrics + calibration JSON files auto-download
8. Copy the `.pt` and `calibration_v2_*.json` into `pavescan-ai/models/`; copy `metrics_v2_*.json` into the repo root

The unselected options will skip automatically — you do not need to pick cells manually.

---

## Three Training Options:

| Option | Dataset | Classes | Images | Type | Time (T4) | Time (A100) | Best For |
|--------|---------|---------|--------|------|-----------|-------------|----------|
| **A** | Crack-Seg | 1 (crack) | 4,029 | Segmentation | ~3-4 hr | ~45-90 min | Pixel-level crack masks |
| **B (Max)** | RDD2022 — 5 countries train, **Japan held out** | 4 | ~47,000 | Detection + cross-domain test | ~10-14 hr | ~3-5 hr | Max diversity + true generalization metric |
| **C (Recommended)** | RDD2022 US+Czech | 4 | ~8,600 | Detection | ~3-4 hr | ~45-90 min | Cold-climate roads (Ontario) |

**Times above reflect v2.1's longer 200-epoch training. Option B exceeds free Colab limits — use Pro or run Option C.**

**Best strategy:** Run the notebook **twice** — once with `TRAINING_OPTION = "A"` and once with `"B"` (or `"C"` if Pro isn't available). The PaveScan AI ensemble runs both models together.


## 1. Setup & GPU Detection

In [ ]:
# === Preflight + Self-Healing Setup ===
# Defines _ps_setup(): an idempotent function that populates the notebook's
# global namespace (via builtins) with every name the rest of the notebook needs.
# Run this cell ONCE per Colab session. After that, every other cell works
# regardless of run order. Re-running is free (microseconds).
import sys
assert sys.version_info >= (3, 10), (
    f"Python 3.10+ required, got {sys.version_info.major}.{sys.version_info.minor}. "
    "In Colab: Runtime > Change runtime type."
)


def _ps_setup():
    """Idempotent setup. Safe to call from any cell, any number of times."""
    import builtins
    g = builtins.__dict__
    if g.get("_PS_READY") is True:
        return

    import subprocess
    def _pip(pkg):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

    try:
        import torch, ultralytics  # noqa: F401
    except ImportError:
        print("Installing ultralytics ...")
        _pip("ultralytics>=8.3.0")
        import torch, ultralytics  # noqa: F401
    try:
        import sklearn  # noqa: F401
    except ImportError:
        print("Installing scikit-learn ...")
        _pip("scikit-learn")
    try:
        import yaml  # noqa: F401
    except ImportError:
        _pip("pyyaml")
    try:
        import cv2  # noqa: F401
    except ImportError:
        _pip("opencv-python")

    import json, glob, shutil, random, urllib.request, zipfile, time
    import xml.etree.ElementTree as ET
    from pathlib import Path
    import numpy as np
    import torch, yaml, cv2
    import matplotlib.pyplot as plt
    from ultralytics import YOLO
    from PIL import Image as PILImage
    from IPython.display import Image, display

    g.update({
        "json": json, "glob": glob, "shutil": shutil, "random": random,
        "urllib": urllib, "zipfile": zipfile, "ET": ET, "Path": Path,
        "np": np, "torch": torch, "YOLO": YOLO, "yaml": yaml, "cv2": cv2,
        "plt": plt, "PILImage": PILImage, "Image": Image, "display": display,
        "time": time,
    })

    # GPU detection (was Cell 4)
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        if gpu_mem >= 38:
            tier, ms_seg, ms_det, bs = "a100", "yolo11l-seg.pt", "yolo11l.pt", 8
        elif gpu_mem >= 22:
            tier, ms_seg, ms_det, bs = "l4",   "yolo11l-seg.pt", "yolo11l.pt", 4
        else:
            tier, ms_seg, ms_det, bs = "t4",   "yolo11m-seg.pt", "yolo11m.pt", 4
        print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)  -> tier={tier}")
    else:
        tier, ms_seg, ms_det, bs = "cpu", "yolo11m-seg.pt", "yolo11m.pt", 2
        print("WARNING: No GPU detected. Training will be very slow.")

    g.update({
        "GPU_TIER": tier,
        "MODEL_SIZE_SEG": ms_seg,
        "MODEL_SIZE_DET": ms_det,
        "BATCH_SIZE": bs,
        "IMGSZ": 1280,
        "SEED": 42,
    })

    # Constants (was Cell 5)
    g["CLASS_MAP"] = {
        "D00": 0, "D10": 1, "D20": 2, "D40": 3,
        "Longitudinal Crack": 0, "Transverse Crack": 1,
        "Alligator Crack": 2, "Pothole": 3,
    }
    g["CLASS_NAMES"] = ["Longitudinal", "Transverse", "Alligator", "Pothole"]
    g["COUNTRIES_ALL"] = ["Japan", "India", "Czech", "Norway", "United_States", "China"]
    g["COUNTRIES_US_CZECH"] = ["United_States", "Czech"]

    random.seed(42); np.random.seed(42); torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    # Patch flags kept for compatibility with downstream guards in this notebook
    g["_PATCH_V1_APPLIED"] = True
    g["_PATCH_V2_APPLIED"] = True
    g["_PATCH_V3_APPLIED"] = True

    g["_PS_READY"] = True
    print(f"_ps_setup OK | tier={tier} | seg={ms_seg} | det={ms_det} | "
          f"batch={bs} | imgsz=1280")


_ps_setup()


In [ ]:
_ps_setup()
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU tier: {GPU_TIER}")
print(f"Segmentation model: {MODEL_SIZE_SEG}")
print(f"Detection model:    {MODEL_SIZE_DET}")
print(f"Batch size:         {BATCH_SIZE}")
print(f"Training resolution: {IMGSZ}px")


In [ ]:
_ps_setup()
print(f"Constants loaded.  CLASS_NAMES = {CLASS_NAMES}")
print(f"COUNTRIES_ALL ({len(COUNTRIES_ALL)}): {COUNTRIES_ALL}")
print(f"COUNTRIES_US_CZECH ({len(COUNTRIES_US_CZECH)}): {COUNTRIES_US_CZECH}")
print(f"SEED = {SEED}  |  IMGSZ = {IMGSZ}")


---

## 2. Pick ONE Training Option

**Set `TRAINING_OPTION` below to `"A"`, `"B"`, or `"C"`. Then click `Runtime → Run All`.** The other two options will skip automatically.

| Option | Output File | Time (T4) | Time (A100) | Best For |
|--------|-------------|-----------|-------------|----------|
| **A** | `pavescan_crack_seg_v2.pt` | ~1 hr | ~20 min | Crack segmentation (default, easiest) |
| **B** | `pavescan_rdd2022_v2.pt` | ~4-6 hr | ~1-2 hr | Maximum accuracy (all 6 countries, ~47K images) |
| **C** | `pavescan_rdd2022_v2.pt` | ~1-2 hr | ~20-30 min | Cold-climate roads (US+Czech, recommended for Ontario) |

**First time?** Leave it on `"A"` — it's the smallest, fastest, and least likely to fail.

In [ ]:
_ps_setup()
SKIP_CALIBRATION = False  # opt out only if you know val data is missing
# =====================================================================
#  PICK ONE: "A", "B", or "C"
# =====================================================================
TRAINING_OPTION = "A"
# =====================================================================

# Resume / fine-tune the partially-trained v2 (epoch 47) instead of training from scratch.
# When True, you must upload pavescan_crack_seg_v2.pt to /content/ before running the fine-tune cell.
RESUME_FROM_V2 = True

# Centralized run name + output filename so downstream cells stay in sync.
RUN_NAME_SEG    = "pavescan_crack_seg_v2_finetune" if RESUME_FROM_V2 else "pavescan_crack_seg_v2"
OUTPUT_NAME_SEG = f"{RUN_NAME_SEG}.pt"

# Friendly summary so you know what's about to happen
_summary = {
    "A": ("Crack Segmentation",       "~2-3 hr A100 fine-tune (35 ep) / ~1 hr from-scratch (T4)", OUTPUT_NAME_SEG),
    "B": ("RDD2022 Full (6 countries)", "~4-6 hours on T4 / ~1-2 hours on A100", "pavescan_rdd2022_v2.pt"),
    "C": ("RDD2022 US+Czech",         "~1-2 hours on T4 / ~20-30 min on A100", "pavescan_rdd2022_v2.pt"),
}

if TRAINING_OPTION not in _summary:
    raise ValueError(
        f"TRAINING_OPTION must be 'A', 'B', or 'C' — got {TRAINING_OPTION!r}. "
        "Edit the cell above and re-run."
    )

_name, _time, _output = _summary[TRAINING_OPTION]
print("=" * 60)
print(f"  Training Option: {TRAINING_OPTION} — {_name}")
print(f"  Expected time:   {_time}")
print(f"  Output file:     {_output}")
if TRAINING_OPTION == "A":
    print(f"  RESUME_FROM_V2:  {RESUME_FROM_V2}  (run dir: runs/segment/{RUN_NAME_SEG})")
print("=" * 60)
print()
print("The other two options will skip automatically.")
print("Keep this Colab tab open and don't let your laptop sleep.")


---

## Confidence Calibration Helpers

These helper functions are used by the per-option calibration cells (after each model finishes training). They fit isotonic regression on the val set so a model's raw `confidence` score maps to actual `P(prediction is a true positive)`. The fitted curve is saved as `calibration_v2_seg.json` (Option A) or `calibration_v2_det.json` (Options B / C) and downloaded alongside the .pt weights.


In [ ]:
# Calibration helpers — used by Option A / B / C calibration cells below.
# Safe to run regardless of TRAINING_OPTION (just defines functions).
_ps_setup()
from sklearn.isotonic import IsotonicRegression
from PIL import Image as _PILImage


def _box_iou(b1, b2):
    """IoU between two xyxy boxes."""
    x1 = max(b1[0], b2[0]); y1 = max(b1[1], b2[1])
    x2 = min(b1[2], b2[2]); y2 = min(b1[3], b2[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    a1 = max(0.0, (b1[2] - b1[0])) * max(0.0, (b1[3] - b1[1]))
    a2 = max(0.0, (b2[2] - b2[0])) * max(0.0, (b2[3] - b2[1]))
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0.0


def _load_yolo_labels(label_path: Path, img_w: int, img_h: int):
    """Read YOLO-format label file. Returns list of (class_id, [x1,y1,x2,y2])."""
    if not label_path.exists():
        return []
    out = []
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            cls = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:5])
        except ValueError:
            continue
        x1 = (cx - bw / 2.0) * img_w
        y1 = (cy - bh / 2.0) * img_h
        x2 = (cx + bw / 2.0) * img_w
        y2 = (cy + bh / 2.0) * img_h
        out.append((cls, [x1, y1, x2, y2]))
    return out


def calibrate_yolo_model(model, val_imgs_dir: Path, val_lbls_dir: Path,
                         imgsz: int = 1280, iou_thresh: float = 0.5,
                         conf_thresh: float = 0.001, max_imgs: int = None) -> dict:
    """Run val inference, match preds to GT by IoU, fit isotonic on confidence -> P(TP).

    Returns a dict with the calibration curve sampled on a uniform grid:
        { method, n_predictions, n_tp, iou_threshold, input_grid, calibrated_grid }
    These can be loaded at inference time and applied via np.interp().
    """
    img_paths = sorted(val_imgs_dir.glob('*.jpg')) + sorted(val_imgs_dir.glob('*.png'))
    if max_imgs is not None:
        img_paths = img_paths[:max_imgs]
    if not img_paths:
        raise RuntimeError(f'No val images found in {val_imgs_dir}')

    confs, is_tp = [], []
    for i, img_path in enumerate(img_paths):
        if i % 200 == 0:
            print(f'  Calibration progress: {i}/{len(img_paths)} images, {len(confs)} preds so far')
        try:
            with _PILImage.open(img_path) as im:
                img_w, img_h = im.size
        except Exception:
            continue
        lbl_path = val_lbls_dir / (img_path.stem + '.txt')
        gts = _load_yolo_labels(lbl_path, img_w, img_h)
        gt_used = [False] * len(gts)
        try:
            results = model.predict(str(img_path), conf=conf_thresh, imgsz=imgsz, verbose=False)
        except Exception:
            continue
        if not results or results[0].boxes is None or len(results[0].boxes) == 0:
            continue
        boxes = results[0].boxes
        n = len(boxes)
        # Sort preds by confidence (descending) so high-conf claims GT first
        ordered = sorted(
            range(n),
            key=lambda j: -float(boxes.conf[j]),
        )
        for j in ordered:
            conf = float(boxes.conf[j])
            cls = int(boxes.cls[j])
            xyxy = boxes.xyxy[j].tolist()
            best_iou = 0.0
            best_idx = -1
            for k, (gcls, gbox) in enumerate(gts):
                if gt_used[k] or gcls != cls:
                    continue
                iou = _box_iou(xyxy, gbox)
                if iou > best_iou:
                    best_iou = iou
                    best_idx = k
            tp = (best_iou >= iou_thresh) and (best_idx >= 0)
            if tp:
                gt_used[best_idx] = True
            confs.append(conf)
            is_tp.append(1 if tp else 0)

    if len(confs) < 50:
        raise RuntimeError(f'Too few predictions ({len(confs)}) for reliable calibration.')

    iso = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
    iso.fit(np.array(confs, dtype=float), np.array(is_tp, dtype=float))
    grid = np.linspace(0.001, 1.0, 41)
    calibrated = iso.predict(grid)

    return {
        'method': 'isotonic',
        'iou_threshold': float(iou_thresh),
        'conf_threshold': float(conf_thresh),
        'n_predictions': int(len(confs)),
        'n_true_positives': int(sum(is_tp)),
        'precision_uncalibrated': float(sum(is_tp) / len(confs)),
        'input_grid': grid.tolist(),
        'calibrated_grid': calibrated.tolist(),
    }


print('Calibration helpers loaded.')


---

## Option A: Crack Segmentation (v2)

The Ultralytics Crack-Seg dataset has **4,029 images** with pixel-level segmentation masks. It auto-downloads when you start training.

- **1 class:** crack
- **Architecture:** YOLO11m-seg (T4) or YOLO11l-seg (A100)
- **Resolution:** 1280px (4x more detail than v1's 640px)
- **Output:** `pavescan_crack_seg_v2.pt`

### Option A — Resume / Fine-tune (NEW)

If `RESUME_FROM_V2 = True` (set in the config cell above), the cell below skips the
from-scratch training and instead **fine-tunes the partially-trained v2 weights**
(epoch 47, paused when Colab disconnected) with gentler hyperparameters:

- Lower LR (0.0008 vs 0.005) — model is already mostly trained, just polish it.
- No mixup, no copy_paste, mosaic 0.5 — these caused EMA NaN/Inf at epoch 49.
- 35 epochs, patience 15 — most of the work is already done.
- Output: `pavescan_crack_seg_v2_finetune.pt` (does NOT overwrite the original).

**Before running:** the cell will prompt you to upload `pavescan_crack_seg_v2.pt`
via `files.upload()` if it isn't already at `/content/`.

**Watch the first 5 epochs.** If you see `WARNING: Skipping checkpoint save ... EMA contains NaN/Inf`,
or box mAP50 drops below 0.65, **stop immediately** (Runtime → Interrupt), drop `lr0` to 0.0004
in this cell, and re-run. Warm-start file stays cached at `/content/`, no re-upload needed.


In [ ]:
_ps_setup()
if TRAINING_OPTION != "A" or not RESUME_FROM_V2:
    print(f"Skipping Option A fine-tune (TRAINING_OPTION={TRAINING_OPTION!r}, RESUME_FROM_V2={RESUME_FROM_V2!r})")
else:
    from pathlib import Path
    print("=" * 60)
    print("  Fine-tuning Option A v2 from epoch-47 best.pt")
    print("=" * 60)
    print("  Expected duration: ~2-3 hours on A100 (35 epochs at 1280px)")
    print("  Strategy: warm-start from your uploaded best.pt with gentler hparams")
    print("  Output run dir: runs/segment/" + RUN_NAME_SEG)
    print("=" * 60)

    # Step 1: locate or upload the warm-start weights
    WARM_START = Path("/content/pavescan_crack_seg_v2.pt")
    if not WARM_START.exists():
        try:
            from google.colab import files
            print("\nUpload pavescan_crack_seg_v2.pt now (211 MB).")
            uploaded = files.upload()
            for fname in uploaded.keys():
                src = Path(fname)
                if src.resolve() != WARM_START.resolve():
                    src.rename(WARM_START)
        except ImportError:
            raise RuntimeError(
                f"Not running in Colab and no warm-start file at {WARM_START}."
            )
    if not WARM_START.exists():
        raise RuntimeError(f"Upload completed but {WARM_START} still missing.")

    size_mb = WARM_START.stat().st_size / 1e6
    print(f"\nWarm-start weights ready: {WARM_START} ({size_mb:.1f} MB)")
    if not (200 < size_mb < 230):
        print(f"  WARNING: expected ~211 MB, got {size_mb:.1f} MB. Continuing.")

    # Step 2: load trained weights as the starting model (NOT resume=True)
    # Why not resume=True: requires a real last.pt with full live optimizer/scheduler/scaler
    # state — we only have best.pt (and even renamed it would misbehave).
    model_seg = YOLO(str(WARM_START))

    # Step 3: fine-tune with gentle hyperparameters
    results_seg = model_seg.train(
        data="crack-seg.yaml",
        epochs=35,                    # trimmed from 50 -- 35 units of compute available
        imgsz=IMGSZ,
        batch=BATCH_SIZE,
        name=RUN_NAME_SEG,
        project="runs/segment",
        exist_ok=True,
        patience=15,
        save=True,
        plots=True,
        optimizer="AdamW",
        seed=SEED,
        lr0=0.0008,                   # was 0.005
        lrf=0.01,
        warmup_epochs=2.0,            # was 5.0
        multi_scale=False,
        amp=False,                    # KEEP — project memory: NaN risk at 1280
        # Augmentation: dial back unstable knobs
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        fliplr=0.5, flipud=0.1,
        degrees=10.0, translate=0.1, scale=0.5, shear=1.0, perspective=0.0,
        mosaic=0.5,                   # was 1.0
        mixup=0.0,                    # was 0.15 — KILL: NaN source
        copy_paste=0.0,               # was 0.15 — KILL: NaN source
        label_smoothing=0.05,
        close_mosaic=10,              # last 10 epochs see clean images
    )

    print("\nOption A fine-tune complete!")
    print(f"Best weights: runs/segment/{RUN_NAME_SEG}/weights/best.pt")


In [ ]:
_ps_setup()
if TRAINING_OPTION != "A":
    print(f"Skipping Option A training (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    print("=" * 60)
    print("  Starting Option A training (Crack Segmentation v2.1)")
    print("=" * 60)
    print("  Expected duration:")
    print("    ~3-4 hours on T4 (free Colab) — 150 epochs at 1280px")
    print("    ~45-90 minutes on A100 (Colab Pro)")
    print()
    print("  Keep this tab open. Don't let your laptop sleep.")
    print("  Free Colab disconnects after ~90 min idle — wiggle the mouse now and then.")
    print("=" * 60)

    # Load the YOLO11 segmentation model (pretrained on COCO)
    model_seg = YOLO(MODEL_SIZE_SEG)

    # v2.1: longer training + aggressive augmentation
    results_seg = model_seg.train(
        data="crack-seg.yaml",            # built-in dataset (auto-downloads ~92 MB)
        epochs=150,                       # v2.1: longer training for max accuracy
        imgsz=IMGSZ,                      # 1280px high resolution
        batch=BATCH_SIZE,                 # auto-set by GPU tier
        name="pavescan_crack_seg_v2",
        project="runs/segment",
        exist_ok=True,
        patience=50,                      # v2.1: more patience for slow convergence
        save=True,
        plots=True,
        optimizer="AdamW",
        seed=SEED,                        # E2: explicit ultralytics seed for reproducibility
        lr0=0.005,                        # E6: was default 0.01; caused EMA NaN/Inf at epoch 3
        warmup_epochs=5.0,                # E6: longer warmup for stability
        # Multi-scale training is OFF — required at small batch (BatchNorm crash on 1x1 features)
        multi_scale=False,  # E6: was True; caused BatchNorm crash on small batch
        amp=False,                        # E7: was default True; EMA NaN/Inf at epoch 14+ (1280px+aggressive aug)
        # v2.1 aggressive augmentation
        hsv_h=0.02, hsv_s=0.8, hsv_v=0.5, # color jitter
        fliplr=0.5, flipud=0.1,           # vertical flip occasionally helps drone imagery
        degrees=15.0,                     # random rotation up to 15 deg
        translate=0.15,
        scale=0.6,
        shear=2.0,
        perspective=0.0005,
        mosaic=1.0,                       # mosaic augmentation (full strength)
        mixup=0.15,                       # blend two images together
        copy_paste=0.15,                  # copy-paste augmentation (seg only)
        label_smoothing=0.05,             # seg: helps generalize
    )

    print("\nOption A training complete!")
    print("Best weights: runs/segment/pavescan_crack_seg_v2/weights/best.pt")


In [ ]:
_ps_setup()
if TRAINING_OPTION != "A":
    print(f"Skipping Option A validation (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    from pathlib import Path

    BEST_PATH = f"runs/segment/{RUN_NAME_SEG}/weights/best.pt"
    if not Path(BEST_PATH).exists():
        raise RuntimeError(
            f"Trained model not found at {BEST_PATH}. "
            "Did the training cell above run successfully?"
        )

    # Evaluate the trained crack segmentation model
    best_seg = YOLO(BEST_PATH)
    metrics_seg = best_seg.val()

    print("\n=== Crack Segmentation v2 Performance ===")
    print(f"Run dir:               runs/segment/{RUN_NAME_SEG}")
    print(f"Segmentation mAP50:    {metrics_seg.seg.map50:.3f}")
    print(f"Segmentation mAP50-95: {metrics_seg.seg.map:.3f}")
    print(f"Box mAP50:             {metrics_seg.box.map50:.3f}")
    print(f"Box mAP50-95:          {metrics_seg.box.map:.3f}")


In [ ]:
_ps_setup()
if TRAINING_OPTION != "A":
    print(f"Skipping Option A visualization (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    from IPython.display import Image, display
    from pathlib import Path

    run_dir = Path(f"runs/segment/{RUN_NAME_SEG}")

    # Show training curves
    if (run_dir / "results.png").exists():
        print("Training curves:")
        display(Image(filename=str(run_dir / "results.png"), width=800))

    # Show sample predictions
    if (run_dir / "val_batch0_pred.jpg").exists():
        print("\nSample predictions:")
        display(Image(filename=str(run_dir / "val_batch0_pred.jpg"), width=800))

    # Show confusion matrix
    if (run_dir / "confusion_matrix.png").exists():
        print("\nConfusion matrix:")
        display(Image(filename=str(run_dir / "confusion_matrix.png"), width=600))


In [ ]:
_ps_setup()
if TRAINING_OPTION != "A":
    print(f"Skipping Option A test sample (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    import glob
    import matplotlib.pyplot as plt
    import cv2

    # Test on a sample image
    test_images = glob.glob("datasets/crack-seg/images/test/*.jpg")
    if test_images:
        test_img = test_images[0]
        results = best_seg.predict(test_img, conf=0.25, imgsz=IMGSZ)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

        original = cv2.imread(test_img)
        original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
        ax1.imshow(original)
        ax1.set_title("Original")
        ax1.axis("off")

        annotated = results[0].plot()
        annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        ax2.imshow(annotated)
        ax2.set_title(f"Detected: {len(results[0].boxes)} cracks")
        ax2.axis("off")

        plt.tight_layout()
        plt.show()
    else:
        print("No test images found")

In [ ]:
_ps_setup()
if TRAINING_OPTION != "A":
    print(f"Skipping Option A v1-vs-v2 comparison (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    import json
    from pathlib import Path

    metrics = {
        "version": "v2_finetune" if RESUME_FROM_V2 else "v2",
        "training_option": "A",
        "model_size": MODEL_SIZE_SEG,
        "imgsz": IMGSZ,
        "batch_size": BATCH_SIZE,
        "epochs_planned": 35 if RESUME_FROM_V2 else 150,
        "patience": 15 if RESUME_FROM_V2 else 50,
        "multi_scale": False,
        "warm_start": RESUME_FROM_V2,
    }

    V2_PATH = f"runs/segment/{RUN_NAME_SEG}/weights/best.pt"
    if not Path(V2_PATH).exists():
        raise RuntimeError(f"v2 weights not found at {V2_PATH}.")

    # --- v2 metrics on Crack-Seg val ---
    print("Evaluating v2 on Crack-Seg val set...")
    v2 = YOLO(V2_PATH)
    v2_val = v2.val(data="crack-seg.yaml", imgsz=IMGSZ, batch=BATCH_SIZE, plots=False, verbose=False)
    metrics["v2_val_seg_mAP50"]    = float(v2_val.seg.map50)
    metrics["v2_val_seg_mAP50_95"] = float(v2_val.seg.map)
    metrics["v2_val_box_mAP50"]    = float(v2_val.box.map50)
    metrics["v2_val_box_mAP50_95"] = float(v2_val.box.map)

    # --- v1 baseline: only runs if user uploaded the v1 .pt to /content/ ---
    V1_PATH = "/content/pavescan_crack_seg.pt"
    if Path(V1_PATH).exists():
        print(f"\nFound v1 baseline at {V1_PATH}. Evaluating on the same val set...")
        try:
            v1 = YOLO(V1_PATH)
            v1_val = v1.val(data="crack-seg.yaml", imgsz=IMGSZ, batch=BATCH_SIZE, plots=False, verbose=False)
            metrics["v1_val_seg_mAP50"]    = float(v1_val.seg.map50)
            metrics["v1_val_seg_mAP50_95"] = float(v1_val.seg.map)
        except Exception as e:
            print(f"v1 evaluation failed (non-fatal): {e}")
            metrics["v1_val_seg_mAP50"]    = None
            metrics["v1_val_seg_mAP50_95"] = None
    else:
        print(f"\nNo v1 baseline at {V1_PATH}. To enable v1-vs-v2 comparison:")
        print("  1. Click the folder icon in Colab's left sidebar")
        print("  2. Drag 'pavescan_crack_seg.pt' from your computer into /content/")
        print("  3. Re-run this single cell (no need to retrain)")
        metrics["v1_val_seg_mAP50"]    = None
        metrics["v1_val_seg_mAP50_95"] = None

    # --- Print comparison table ---
    print("\n" + "=" * 60)
    print("  v1 vs v2 — Crack Segmentation")
    print("=" * 60)
    if metrics["v1_val_seg_mAP50"] is not None:
        d50 = metrics["v2_val_seg_mAP50"] - metrics["v1_val_seg_mAP50"]
        d95 = metrics["v2_val_seg_mAP50_95"] - metrics["v1_val_seg_mAP50_95"]
        print(f"  Seg mAP50:    v1={metrics['v1_val_seg_mAP50']:.3f}  ->  v2={metrics['v2_val_seg_mAP50']:.3f}  (delta {d50:+.3f})")
        print(f"  Seg mAP50-95: v1={metrics['v1_val_seg_mAP50_95']:.3f}  ->  v2={metrics['v2_val_seg_mAP50_95']:.3f}  (delta {d95:+.3f})")
    else:
        print(f"  Seg mAP50:    v2={metrics['v2_val_seg_mAP50']:.3f}    (v1 not evaluated)")
        print(f"  Seg mAP50-95: v2={metrics['v2_val_seg_mAP50_95']:.3f}  (v1 not evaluated)")
    print("=" * 60)

    # --- Save metrics JSON for the README ---
    metrics_filename = f"metrics_v2_seg{'_finetune' if RESUME_FROM_V2 else ''}.json"
    metrics_path = Path(metrics_filename)
    metrics_path.write_text(json.dumps(metrics, indent=2))
    print(f"\nMetrics saved to: {metrics_path}")
    print("This file will download alongside your model — drop it in the repo root.")


In [ ]:
# Confidence calibration for Option A (Crack-Seg)
_ps_setup()
if TRAINING_OPTION != 'A':
    print(f"Skipping Option A calibration (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    from pathlib import Path
    import json

    V2_PATH = f"runs/segment/{RUN_NAME_SEG}/weights/best.pt"
    if not Path(V2_PATH).exists():
        raise RuntimeError(f'v2 weights not found at {V2_PATH}.')

    # Locate Crack-Seg val images + labels (auto-downloaded by ultralytics)
    candidates = list(Path('datasets').rglob('crack-seg/images/val'))
    if not candidates:
        candidates = list(Path('datasets').rglob('images/val'))
    val_imgs_dir = candidates[0] if candidates else Path('datasets/crack-seg/images/val')
    val_lbls_dir = val_imgs_dir.parent.parent / 'labels' / 'val'

    if not val_imgs_dir.exists() or not val_lbls_dir.exists():
        if SKIP_CALIBRATION:
            print(f'SKIP_CALIBRATION=True; skipping calibration (would have used {val_imgs_dir})')
        else:
            raise RuntimeError(
                f'Calibration val data not found.\n'
                f'  Tried images: {val_imgs_dir}\n'
                f'  Tried labels: {val_lbls_dir}\n'
                f'If training succeeded, the val set should be on disk. '
                f'To skip calibration anyway, set SKIP_CALIBRATION = True in the config cell.'
            )
    else:
        print(f'Calibrating against {val_imgs_dir} ({sum(1 for _ in val_imgs_dir.glob("*.jpg"))} images)')
        v2_model = YOLO(V2_PATH)
        cal = calibrate_yolo_model(
            v2_model,
            val_imgs_dir=val_imgs_dir,
            val_lbls_dir=val_lbls_dir,
            imgsz=IMGSZ,
        )
        cal['version'] = 'v2_finetune' if RESUME_FROM_V2 else 'v2'
        cal['training_option'] = 'A'
        cal['model'] = RUN_NAME_SEG
        cal['model_size'] = MODEL_SIZE_SEG

        cal_filename = f"calibration_v2_seg{'_finetune' if RESUME_FROM_V2 else ''}.json"
        out_path = Path(cal_filename)
        out_path.write_text(json.dumps(cal, indent=2))
        print(f"\nCalibration saved to: {out_path}")
        print(f"  n_predictions = {cal['n_predictions']}, raw precision = {cal['precision_uncalibrated']:.3f}")
        print(f'  Drop this file in your repo at: pavescan-ai/models/{cal_filename}')


In [ ]:
_ps_setup()
if TRAINING_OPTION != "A":
    print(f"Skipping Option A download (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    # _E7_GUARDED — wrap so the cell still works locally
    try:
        from google.colab import files
        _COLAB_OK = True
    except ImportError:
        _COLAB_OK = False
    from pathlib import Path
    import shutil

    BEST_PATH = f"runs/segment/{RUN_NAME_SEG}/weights/best.pt"
    OUTPUT_NAME = OUTPUT_NAME_SEG

    if not Path(BEST_PATH).exists():
        raise RuntimeError(
            f"Trained weights not found at {BEST_PATH}. "
            "Cannot download — did training succeed?"
        )

    shutil.copy(BEST_PATH, OUTPUT_NAME)

    print(f"Downloading {OUTPUT_NAME} to your computer...")
    print(f"After download finishes, copy this file to: pavescan-ai/models/{OUTPUT_NAME}")
    if _COLAB_OK:
        files.download(OUTPUT_NAME)
    else:
        print("  (not in Colab — skipping download)")

    metrics_filename = f"metrics_v2_seg{'_finetune' if RESUME_FROM_V2 else ''}.json"
    metrics_file = Path(metrics_filename)
    if metrics_file.exists():
        print(f"\nDownloading {metrics_file.name} (drop in your repo root)...")
        if _COLAB_OK:
            files.download(str(metrics_file))
        else:
            print("  (not in Colab — skipping download)")

    cal_filename = f"calibration_v2_seg{'_finetune' if RESUME_FROM_V2 else ''}.json"
    calibration_file = Path(cal_filename)
    if calibration_file.exists():
        print(f"\nDownloading {calibration_file.name} (drop in pavescan-ai/models/)...")
        if _COLAB_OK:
            files.download(str(calibration_file))
        else:
            print("  (not in Colab — skipping download)")


---

## Option B: RDD2022 — Train on 5 Countries, Hold Out Japan (Cross-Domain Validation)

The **Road Damage Dataset 2022** has **~47,000 images** from 6 countries. This option trains on **5 of them** and holds out **Japan** as a cross-domain test set — measuring whether the model truly generalizes to unseen road geographies, or just memorizes one region's surface textures.

- **4 classes:** Longitudinal Crack, Transverse Crack, Alligator Crack, Pothole
- **Train + val pool (5 countries):** India, Czech Republic, Norway, United States, China (~37k images)
- **Held-out test (Japan):** ~10k images, never seen during training
- **Total download:** ~3.7 GB
- **Architecture:** YOLO11m (T4) or YOLO11l (A100)
- **Resolution:** 1280px
- **Training time:** ~10-14 hours (T4, exceeds free Colab limits) or ~3-5 hours (A100, Pro)
- **Outputs:** `pavescan_rdd2022_v2.pt`, `metrics_v2_det.json`, `metrics_v2_japan_holdout.json`, `calibration_v2_det.json`

> **Why hold out Japan?** Random 85/15 splits leak texture, lighting, and signage from each country into both train and val, so val mAP overstates real-world performance. A geographic holdout is the standard cross-domain generalization test in published RDD2022 benchmarks. Even a small drop on the held-out country (vs in-domain val) is the honest number to report.

> **Note:** This is a detection model (bounding boxes), not segmentation. Your PaveScan AI ensemble runs both this and the segmentation model together.


In [ ]:
_ps_setup()
if TRAINING_OPTION != "B":
    print(f"Skipping Option B download (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    # _E7_GUARDED — wrap so the cell still works locally
    try:
        from google.colab import files
        _COLAB_OK = True
    except ImportError:
        _COLAB_OK = False
    from pathlib import Path
    import shutil

    BEST_PATH = "runs/detect/pavescan_rdd2022_all_v2/weights/best.pt"
    OUTPUT_NAME = "pavescan_rdd2022_v2.pt"

    if not Path(BEST_PATH).exists():
        raise RuntimeError(
            f"Trained weights not found at {BEST_PATH}. "
            "Cannot download — did training succeed?"
        )

    shutil.copy(BEST_PATH, OUTPUT_NAME)
    print(f"Downloading {OUTPUT_NAME} to your computer...")
    print(f"After download finishes, copy this file to: pavescan-ai/models/{OUTPUT_NAME}")
    if _COLAB_OK:
        files.download(OUTPUT_NAME)
    else:
        print("  (not in Colab — skipping download)")

    metrics_file = Path("metrics_v2_det.json")
    if metrics_file.exists():
        print(f"\nDownloading {metrics_file.name} (drop in your repo root)...")
        if _COLAB_OK:
            files.download(str(metrics_file))
        else:
            print("  (not in Colab — skipping download)")

    holdout_file = Path("metrics_v2_japan_holdout.json")
    if holdout_file.exists():
        print(f"\nDownloading {holdout_file.name} (drop in your repo root)...")
        if _COLAB_OK:
            files.download(str(holdout_file))
        else:
            print("  (not in Colab — skipping download)")

    calibration_file = Path("calibration_v2_det.json")
    if calibration_file.exists():
        print(f"\nDownloading {calibration_file.name} (drop in pavescan-ai/models/)...")
        if _COLAB_OK:
            files.download(str(calibration_file))
        else:
            print("  (not in Colab — skipping download)")


In [ ]:
# === RDD2022 Download (with retry + integrity check) ===
# Required by Options B and C. Skipped if TRAINING_OPTION = "A".
# Idempotent: re-running just verifies extracted folders are present.
_ps_setup()
if TRAINING_OPTION not in ("B", "C"):
    print(f"Skipping RDD2022 download (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    raw_dir = Path("rdd2022_raw")
    raw_dir.mkdir(exist_ok=True)
    BASE = "https://github.com/sekilab/RoadDamageDetector/releases/download/RDD2022_for_CRDDC2022"
    wanted = COUNTRIES_ALL if TRAINING_OPTION == "B" else COUNTRIES_US_CZECH

    def _download_with_retry(url, dest, max_attempts=3):
        last = None
        for attempt in range(1, max_attempts + 1):
            try:
                print(f"    attempt {attempt}/{max_attempts} ...")
                urllib.request.urlretrieve(url, dest)
                size_mb = dest.stat().st_size / 1e6
                if size_mb < 1:
                    raise RuntimeError(f"file suspiciously small ({size_mb:.1f} MB)")
                if not zipfile.is_zipfile(dest):
                    raise RuntimeError("downloaded file is not a valid zip")
                print(f"    OK ({size_mb:.0f} MB, integrity verified)")
                return
            except Exception as e:
                last = e
                print(f"    attempt {attempt} failed: {e}")
                if dest.exists():
                    dest.unlink()
                if attempt < max_attempts:
                    time.sleep(5 * attempt)
        raise RuntimeError(
            f"Failed to download {url} after {max_attempts} attempts.\n"
            f"  Last error: {last}\n"
            f"  Manual fallback:  curl -LO {url}"
        )

    for country in wanted:
        if any(raw_dir.rglob(f"*{country}*/train/images")):
            print(f"  {country}: already extracted, skipping.")
            continue
        zip_path = raw_dir / f"RDD2022_{country}.zip"
        if zip_path.exists() and not zipfile.is_zipfile(zip_path):
            print(f"  {country}: corrupt cached zip; deleting and retrying.")
            zip_path.unlink()
        if not zip_path.exists():
            url = f"{BASE}/RDD2022_{country}.zip"
            print(f"  Downloading {country} from {url}")
            _download_with_retry(url, zip_path)
        print(f"  Extracting {country} ...")
        try:
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(raw_dir)
        except zipfile.BadZipFile as e:
            zip_path.unlink(missing_ok=True)
            raise RuntimeError(f"Bad zip for {country}: {e}. Re-run this cell.") from e

    missing = [c for c in wanted if not any(raw_dir.rglob(f"*{c}*/train/images"))]
    if missing:
        raise RuntimeError(
            f"FATAL: missing train/images for {missing} after download.\n"
            f"raw_dir contents: {list(raw_dir.iterdir())}"
        )
    print(f"\nAll {len(wanted)} countries present in {raw_dir}/")


In [ ]:
_ps_setup()
if TRAINING_OPTION != "B":
    print(f"Skipping Option B conversion (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    import xml.etree.ElementTree as ET
    from pathlib import Path
    from PIL import Image as PILImage
    import shutil
    import random

    # --- Convert PascalVOC to YOLO format ---
    def convert_voc_to_yolo(xml_path, img_width, img_height):
        """Convert a PascalVOC XML annotation to YOLO format lines."""
        tree = ET.parse(xml_path)
        root = tree.getroot()
        lines = []
        for obj in root.findall("object"):
            label = obj.find("name").text
            if label not in CLASS_MAP:
                continue
            class_id = CLASS_MAP[label]
            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            cx = (xmin + xmax) / 2.0 / img_width
            cy = (ymin + ymax) / 2.0 / img_height
            w = (xmax - xmin) / img_width
            h = (ymax - ymin) / img_height
            lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
        return lines

    # --- Collect pairs PER COUNTRY (so we can hold out Japan) ---
    raw_dir = Path("rdd2022_raw")
    pairs_by_country = {}

    for country in COUNTRIES_ALL:
        img_dirs = list(raw_dir.rglob(f"*{country}*/train/images"))
        xml_dirs = list(raw_dir.rglob(f"*{country}*/train/annotations/xmls"))

        if not img_dirs or not xml_dirs:
            raise RuntimeError(f"FATAL: Could not find RDD2022 data for {country} under {raw_dir}/. Did the download cell run successfully?")

        img_dir = img_dirs[0]
        xml_dir = xml_dirs[0]

        country_pairs = []
        for xml_file in sorted(xml_dir.glob("*.xml")):
            img_file = img_dir / xml_file.name.replace(".xml", ".jpg")
            if img_file.exists():
                country_pairs.append((img_file, xml_file))

        pairs_by_country[country] = country_pairs
        print(f"{country}: {len(country_pairs)} image-annotation pairs")

    # --- Hold Japan out as a cross-domain test set ---
    HOLDOUT_COUNTRY = "Japan"
    japan_pairs = pairs_by_country.pop(HOLDOUT_COUNTRY, [])
    train_val_pairs = []
    for c, pairs in pairs_by_country.items():
        train_val_pairs.extend(pairs)

    print()
    print(f"  Holdout country: {HOLDOUT_COUNTRY} ({len(japan_pairs)} images, never seen in training)")
    print(f"  Training pool (5 countries): {len(train_val_pairs)} images")

    if len(train_val_pairs) == 0:
        raise RuntimeError("No training data found! Check the downloads above for errors.")

    # --- Shuffle and split: 85% train, 15% val (5 countries only) ---
    random.seed(42)
    random.shuffle(train_val_pairs)
    split = int(0.85 * len(train_val_pairs))
    train_pairs = train_val_pairs[:split]
    val_pairs = train_val_pairs[split:]

    out_dir = Path("datasets/rdd2022_all")

    splits = [
        ("train", train_pairs),
        ("val", val_pairs),
        ("test_japan", japan_pairs),
    ]

    for split_name, split_pairs in splits:
        img_out = out_dir / "images" / split_name
        lbl_out = out_dir / "labels" / split_name
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)

        skipped = 0
        for img_path, xml_path in split_pairs:
            img = PILImage.open(img_path)
            w, h = img.size
            yolo_lines = convert_voc_to_yolo(xml_path, w, h)
            if not yolo_lines:
                skipped += 1
                continue
            country_prefix = img_path.parts[-4] if len(img_path.parts) > 4 else "unknown"
            out_name = f"{country_prefix}_{img_path.name}"
            shutil.copy(img_path, img_out / out_name)
            label_file = lbl_out / out_name.replace(".jpg", ".txt")
            label_file.write_text("\n".join(yolo_lines))

        total = len(list(img_out.glob("*.jpg")))
        print(f"{split_name}: {total} images ({skipped} skipped)")

    # --- Final sanity check ---
    train_count = len(list((out_dir / "images" / "train").glob("*.jpg")))
    val_count   = len(list((out_dir / "images" / "val").glob("*.jpg")))
    test_count  = len(list((out_dir / "images" / "test_japan").glob("*.jpg")))

    if train_count < 1000 or val_count < 100:
        raise RuntimeError(
            f"Conversion produced too few usable images: "
            f"train={train_count}, val={val_count}. Cannot train."
        )
    if test_count < 100:
        print(f"WARNING: Japan holdout has only {test_count} images -- cross-domain eval may be noisy.")

    print(f"\nDataset ready! train={train_count}, val={val_count}, test_japan={test_count}")
    print("Train+val pool excludes Japan -- enables true cross-domain generalization measurement.")


In [ ]:
_ps_setup()
if TRAINING_OPTION != "B":
    print(f"Skipping Option B training (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    import yaml
    from pathlib import Path

    print("=" * 60)
    print("  Starting Option B training (RDD2022 v2.1 — 5 countries, Japan held out)")
    print("=" * 60)
    print("  Expected duration:")
    print("    ~10-14 hours on T4 — WILL EXCEED free Colab session limits")
    print("    ~3-5 hours on A100 (Colab Pro strongly recommended)")
    print()
    print("  Free Colab disconnects after ~12 hours of activity OR ~90 min idle.")
    print("  Strongly recommend Colab Pro / A100 for this option.")
    print("  Keep this tab open. Don't let your laptop sleep.")
    print()
    print("  Note: Japan is held out as cross-domain test set -- not used for training/val.")
    print("=" * 60)

    # Training dataset config (5 countries, train + val splits)
    dataset_config = {
        "path": str(Path("datasets/rdd2022_all").resolve()),
        "train": "images/train",
        "val": "images/val",
        "names": {i: name for i, name in enumerate(CLASS_NAMES)},
    }

    config_path = Path("rdd2022_all.yaml")
    config_path.write_text(yaml.dump(dataset_config, default_flow_style=False))
    print("Training dataset config:")
    print(config_path.read_text())

    # Japan holdout config (used by cross-domain evaluation cell below)
    japan_config = {
        "path": str(Path("datasets/rdd2022_all").resolve()),
        "train": "images/train",      # required by ultralytics yaml schema
        "val": "images/test_japan",   # point val to Japan holdout
        "names": {i: name for i, name in enumerate(CLASS_NAMES)},
    }
    japan_config_path = Path("rdd2022_japan_holdout.yaml")
    japan_config_path.write_text(yaml.dump(japan_config, default_flow_style=False))

    # Train YOLO11 on RDD2022 (5 countries) with v2.1 settings
    model_rdd_full = YOLO(MODEL_SIZE_DET)

    results_rdd_full = model_rdd_full.train(
        data="rdd2022_all.yaml",
        epochs=150,                       # v2.1: longer training
        imgsz=IMGSZ,
        batch=BATCH_SIZE,
        name="pavescan_rdd2022_all_v2",
        project="runs/detect",
        exist_ok=True,
        patience=50,                      # v2.1: more patience
        save=True,
        plots=True,
        optimizer="AdamW",
        seed=SEED,                        # E2: explicit ultralytics seed for reproducibility
        lr0=0.005,                        # E6: was default 0.01; caused EMA NaN/Inf at epoch 3
        warmup_epochs=5.0,                # E6: longer warmup for stability
        multi_scale=False,  # E6: was True; caused BatchNorm crash on small batch
        amp=False,                        # E7: was default True; EMA NaN/Inf at epoch 14+ (1280px+aggressive aug)
        # v2.1 aggressive augmentation (det — no copy_paste, no label_smoothing)
        hsv_h=0.02, hsv_s=0.8, hsv_v=0.5,
        fliplr=0.5, flipud=0.1,
        degrees=15.0,
        translate=0.15,
        scale=0.6,
        shear=2.0,
        perspective=0.0005,
        mosaic=1.0,
        mixup=0.15,
        # Focal loss to focus on rare classes (Pothole D40)
        fl_gamma=1.5,
    )

    print("\nOption B training complete!")
    print("Best weights: runs/detect/pavescan_rdd2022_all_v2/weights/best.pt")


In [ ]:
_ps_setup()
if TRAINING_OPTION != "B":
    print(f"Skipping Option B validation (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    from pathlib import Path

    BEST_PATH = "runs/detect/pavescan_rdd2022_all_v2/weights/best.pt"
    if not Path(BEST_PATH).exists():
        raise RuntimeError(
            f"Trained model not found at {BEST_PATH}. "
            "Did training succeed?"
        )

    best_rdd = YOLO(BEST_PATH)
    metrics_rdd = best_rdd.val()

    print("\n=== RDD2022 Full v2 Performance ===")
    print(f"mAP50:    {metrics_rdd.box.map50:.3f}")
    print(f"mAP50-95: {metrics_rdd.box.map:.3f}")

    for i, name in enumerate(CLASS_NAMES):
        if i < len(metrics_rdd.box.ap50):
            print(f"  {name}: AP50 = {metrics_rdd.box.ap50[i]:.3f}")

In [ ]:
_ps_setup()
if TRAINING_OPTION != "B":
    print(f"Skipping Option B v1-vs-v2 comparison (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    import json
    from pathlib import Path

    metrics = {
        "version": "v2",
        "training_option": "B",
        "dataset": "RDD2022_all_6_countries",
        "model_size": MODEL_SIZE_DET,
        "imgsz": IMGSZ,
        "batch_size": BATCH_SIZE,
        "epochs_planned": 150,
        "patience": 50,
        "multi_scale": False,
        "fl_gamma": 1.5,
        "class_names": CLASS_NAMES,
    }

    V2_PATH = "runs/detect/pavescan_rdd2022_all_v2/weights/best.pt"
    if not Path(V2_PATH).exists():
        raise RuntimeError(f"v2 weights not found at {V2_PATH}.")

    # --- v2 metrics on Option B val ---
    print("Evaluating v2 on RDD2022 (all-6-countries) val set...")
    v2 = YOLO(V2_PATH)
    v2_val = v2.val(data="rdd2022_all.yaml", imgsz=IMGSZ, batch=BATCH_SIZE, plots=False, verbose=False)
    metrics["v2_val_box_mAP50"]    = float(v2_val.box.map50)
    metrics["v2_val_box_mAP50_95"] = float(v2_val.box.map)
    metrics["v2_val_per_class_AP50"] = {
        CLASS_NAMES[i]: float(v2_val.box.ap50[i])
        for i in range(min(len(CLASS_NAMES), len(v2_val.box.ap50)))
    }

    # --- v1 baseline ---
    V1_PATH = "/content/pavescan_rdd2022.pt"
    if Path(V1_PATH).exists():
        print(f"\nFound v1 baseline at {V1_PATH}. Evaluating on the same val set...")
        try:
            v1 = YOLO(V1_PATH)
            v1_val = v1.val(data="rdd2022_all.yaml", imgsz=IMGSZ, batch=BATCH_SIZE, plots=False, verbose=False)
            metrics["v1_val_box_mAP50"]    = float(v1_val.box.map50)
            metrics["v1_val_box_mAP50_95"] = float(v1_val.box.map)
        except Exception as e:
            print(f"v1 evaluation failed (non-fatal): {e}")
            metrics["v1_val_box_mAP50"]    = None
            metrics["v1_val_box_mAP50_95"] = None
    else:
        print(f"\nNo v1 baseline at {V1_PATH}. To enable v1-vs-v2 comparison:")
        print("  1. Click the folder icon in Colab's left sidebar")
        print("  2. Drag 'pavescan_rdd2022.pt' from your computer into /content/")
        print("  3. Re-run this single cell")
        metrics["v1_val_box_mAP50"]    = None
        metrics["v1_val_box_mAP50_95"] = None

    # --- Print comparison ---
    print("\n" + "=" * 60)
    print("  v1 vs v2 — RDD2022 (all 6 countries)")
    print("=" * 60)
    if metrics["v1_val_box_mAP50"] is not None:
        d50 = metrics["v2_val_box_mAP50"] - metrics["v1_val_box_mAP50"]
        d95 = metrics["v2_val_box_mAP50_95"] - metrics["v1_val_box_mAP50_95"]
        print(f"  Box mAP50:    v1={metrics['v1_val_box_mAP50']:.3f}  ->  v2={metrics['v2_val_box_mAP50']:.3f}  (delta {d50:+.3f})")
        print(f"  Box mAP50-95: v1={metrics['v1_val_box_mAP50_95']:.3f}  ->  v2={metrics['v2_val_box_mAP50_95']:.3f}  (delta {d95:+.3f})")
    else:
        print(f"  Box mAP50:    v2={metrics['v2_val_box_mAP50']:.3f}    (v1 not evaluated)")
        print(f"  Box mAP50-95: v2={metrics['v2_val_box_mAP50_95']:.3f}  (v1 not evaluated)")
    print("\n  Per-class AP50:")
    for cls, ap in metrics["v2_val_per_class_AP50"].items():
        print(f"    {cls:20s}  {ap:.3f}")
    print("=" * 60)

    # --- Save metrics JSON ---
    metrics_path = Path("metrics_v2_det.json")
    metrics_path.write_text(json.dumps(metrics, indent=2))
    print(f"\nMetrics saved to: {metrics_path}")


In [ ]:
# Cross-domain evaluation: how does the 5-country model do on held-out Japan?
_ps_setup()
if TRAINING_OPTION != "B":
    print(f"Skipping Option B Japan-holdout evaluation (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    import json
    from pathlib import Path

    V2_PATH = "runs/detect/pavescan_rdd2022_all_v2/weights/best.pt"
    if not Path(V2_PATH).exists():
        raise RuntimeError(f"v2 weights not found at {V2_PATH}.")

    print("=" * 60)
    print("  Cross-Domain Evaluation: Japan (held out, never seen in training)")
    print("=" * 60)

    v2 = YOLO(V2_PATH)
    japan_val = v2.val(
        data="rdd2022_japan_holdout.yaml",
        imgsz=IMGSZ,
        batch=BATCH_SIZE,
        plots=False,
        verbose=False,
    )

    holdout_metrics = {
        "version": "v2",
        "training_option": "B",
        "holdout_country": "Japan",
        "trained_on_countries": ["India", "Czech", "Norway", "United_States", "China"],
        "imgsz": IMGSZ,
        "japan_holdout_box_mAP50":    float(japan_val.box.map50),
        "japan_holdout_box_mAP50_95": float(japan_val.box.map),
        "japan_holdout_per_class_AP50": {
            CLASS_NAMES[i]: float(japan_val.box.ap50[i])
            for i in range(min(len(CLASS_NAMES), len(japan_val.box.ap50)))
        },
    }

    # Re-evaluate in-domain (5-country val) for delta context
    in_domain_yaml = "rdd2022_all.yaml"
    if Path(in_domain_yaml).exists():
        print("\nRunning in-domain val (5-country mix) for comparison...")
        in_domain = v2.val(data=in_domain_yaml, imgsz=IMGSZ, batch=BATCH_SIZE, plots=False, verbose=False)
        holdout_metrics["in_domain_box_mAP50"]    = float(in_domain.box.map50)
        holdout_metrics["in_domain_box_mAP50_95"] = float(in_domain.box.map)

    print("\n" + "=" * 60)
    print("  Cross-Domain Generalization Report")
    print("=" * 60)
    if "in_domain_box_mAP50" in holdout_metrics:
        print(f"  In-domain val (5 countries):  mAP50 = {holdout_metrics['in_domain_box_mAP50']:.3f}")
    print(f"  Out-of-domain (Japan):        mAP50 = {holdout_metrics['japan_holdout_box_mAP50']:.3f}")
    if "in_domain_box_mAP50" in holdout_metrics:
        gap = holdout_metrics["in_domain_box_mAP50"] - holdout_metrics["japan_holdout_box_mAP50"]
        print(f"  Generalization gap:           {gap:+.3f}")
        print("  (smaller gap = stronger cross-cultural transfer)")
    print("\n  Per-class AP50 on Japan:")
    for cls, ap in holdout_metrics["japan_holdout_per_class_AP50"].items():
        print(f"    {cls:20s}  {ap:.3f}")
    print("=" * 60)

    out_path = Path("metrics_v2_japan_holdout.json")
    out_path.write_text(json.dumps(holdout_metrics, indent=2))
    print(f"\nSaved cross-domain metrics: {out_path}")
    print("  Drop this file in your repo root alongside metrics_v2_det.json.")


In [ ]:
# Confidence calibration for Option B (RDD2022 — 5 countries, Japan held out)
_ps_setup()
if TRAINING_OPTION != 'B':
    print(f"Skipping Option B calibration (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    from pathlib import Path
    import json

    V2_PATH = 'runs/detect/pavescan_rdd2022_all_v2/weights/best.pt'
    if not Path(V2_PATH).exists():
        raise RuntimeError(f'v2 weights not found at {V2_PATH}.')

    val_imgs_dir = Path('datasets/rdd2022_all/images/val')
    val_lbls_dir = Path('datasets/rdd2022_all/labels/val')
    if not val_imgs_dir.exists() or not val_lbls_dir.exists():
        if SKIP_CALIBRATION:
            print(f'SKIP_CALIBRATION=True; skipping calibration (would have used {val_imgs_dir})')
        else:
            raise RuntimeError(
                f'Calibration val data not found.\n'
                f'  Tried images: {val_imgs_dir}\n'
                f'  Tried labels: {val_lbls_dir}\n'
                f'If training succeeded, the val set should be on disk. '
                f'To skip calibration anyway, set SKIP_CALIBRATION = True in cell 7.'
            )
    else:
        print(f'Calibrating against {val_imgs_dir} ({sum(1 for _ in val_imgs_dir.glob("*.jpg"))} images)')
        v2_model = YOLO(V2_PATH)
        cal = calibrate_yolo_model(
            v2_model,
            val_imgs_dir=val_imgs_dir,
            val_lbls_dir=val_lbls_dir,
            imgsz=IMGSZ,
        )
        cal['version'] = 'v2'
        cal['training_option'] = 'B'
        cal['model'] = 'pavescan_rdd2022_all_v2'
        cal['model_size'] = MODEL_SIZE_DET
        cal['class_names'] = CLASS_NAMES
        cal['holdout_country'] = 'Japan'

        out_path = Path('calibration_v2_det.json')
        out_path.write_text(json.dumps(cal, indent=2))
        print(f"\nCalibration saved to: {out_path}")
        print(f"  n_predictions = {cal['n_predictions']}, raw precision = {cal['precision_uncalibrated']:.3f}")
        print('  Drop this file in your repo at: pavescan-ai/models/calibration_v2_det.json')


In [ ]:
_ps_setup()
if TRAINING_OPTION != "B":
    print(f"Skipping Option B download (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    # _E7_GUARDED — wrap so the cell still works locally
    try:
        from google.colab import files
        _COLAB_OK = True
    except ImportError:
        _COLAB_OK = False
    from pathlib import Path
    import shutil

    BEST_PATH = "runs/detect/pavescan_rdd2022_all_v2/weights/best.pt"
    OUTPUT_NAME = "pavescan_rdd2022_v2.pt"

    if not Path(BEST_PATH).exists():
        raise RuntimeError(
            f"Trained weights not found at {BEST_PATH}. "
            "Cannot download — did training succeed?"
        )

    shutil.copy(BEST_PATH, OUTPUT_NAME)
    print(f"Downloading {OUTPUT_NAME} to your computer...")
    print(f"After download finishes, copy this file to: pavescan-ai/models/{OUTPUT_NAME}")
    if _COLAB_OK:
        files.download(OUTPUT_NAME)
    else:
        print("  (not in Colab — skipping download)")

    metrics_file = Path("metrics_v2_det.json")
    if metrics_file.exists():
        print(f"\nDownloading {metrics_file.name} (drop in your repo root)...")
        if _COLAB_OK:
            files.download(str(metrics_file))
        else:
            print("  (not in Colab — skipping download)")


---

## Option C: US + Czech Republic (Recommended for Ontario)

**Best match for Ontario/Toronto road conditions** — trained on cold-climate road data.

- **Data:** US (~6,005 images) + Czech Republic (~2,629 images) = ~8,600 images
- **4 classes:** Longitudinal Crack, Transverse Crack, Alligator Crack, Pothole
- **Why US + Czech?**
  - **US:** Same AASHTO road construction standards as Ontario/Canada. Northern US states have identical freeze-thaw winters.
  - **Czech Republic:** Cold winters with freeze-thaw cycles and road salt damage — same crack patterns as Toronto roads.
- **Architecture:** YOLO11m (T4) or YOLO11l (A100)
- **Resolution:** 1280px
- **Output:** `pavescan_rdd2022_v2.pt`

In [ ]:
_ps_setup()
if TRAINING_OPTION != "C":
    print(f"Skipping Option C download (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    # _E7_GUARDED — wrap so the cell still works locally
    try:
        from google.colab import files
        _COLAB_OK = True
    except ImportError:
        _COLAB_OK = False
    from pathlib import Path
    import shutil

    BEST_PATH = "runs/detect/pavescan_rdd2022_uscz_v2/weights/best.pt"
    OUTPUT_NAME = "pavescan_rdd2022_v2.pt"

    if not Path(BEST_PATH).exists():
        raise RuntimeError(
            f"Trained weights not found at {BEST_PATH}. "
            "Cannot download — did training succeed?"
        )

    shutil.copy(BEST_PATH, OUTPUT_NAME)
    print(f"Downloading {OUTPUT_NAME} to your computer...")
    print(f"After download finishes, copy this file to: pavescan-ai/models/{OUTPUT_NAME}")
    if _COLAB_OK:
        files.download(OUTPUT_NAME)
    else:
        print("  (not in Colab — skipping download)")

    metrics_file = Path("metrics_v2_det.json")
    if metrics_file.exists():
        print(f"\nDownloading {metrics_file.name} (drop in your repo root)...")
        if _COLAB_OK:
            files.download(str(metrics_file))
        else:
            print("  (not in Colab — skipping download)")

    calibration_file = Path("calibration_v2_det.json")
    if calibration_file.exists():
        print(f"\nDownloading {calibration_file.name} (drop in pavescan-ai/models/)...")
        if _COLAB_OK:
            files.download(str(calibration_file))
        else:
            print("  (not in Colab — skipping download)")


In [ ]:
_ps_setup()
if TRAINING_OPTION != "C":
    print(f"Skipping Option C conversion (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    import xml.etree.ElementTree as ET
    from pathlib import Path
    from PIL import Image as PILImage
    import shutil
    import random

    # --- Convert VOC to YOLO format ---
    def convert_voc_to_yolo(xml_path, img_width, img_height):
        tree = ET.parse(xml_path)
        root = tree.getroot()
        lines = []
        for obj in root.findall("object"):
            label = obj.find("name").text
            if label not in CLASS_MAP:
                continue
            class_id = CLASS_MAP[label]
            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            cx = (xmin + xmax) / 2.0 / img_width
            cy = (ymin + ymax) / 2.0 / img_height
            w = (xmax - xmin) / img_width
            h = (ymax - ymin) / img_height
            lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
        return lines

    # --- Collect pairs from US + Czech ---
    raw_dir = Path("rdd2022_raw")
    all_pairs = []

    for country in COUNTRIES_US_CZECH:
        img_dirs = list(raw_dir.rglob(f"*{country}*/train/images"))
        xml_dirs = list(raw_dir.rglob(f"*{country}*/train/annotations/xmls"))

        if not img_dirs or not xml_dirs:
            print(f"WARNING: Could not find data for {country}")
            print(f"  Contents of rdd2022_raw/: {[p.name for p in raw_dir.iterdir()]}")
            continue

        img_dir = img_dirs[0]
        xml_dir = xml_dirs[0]

        count = 0
        for xml_file in sorted(xml_dir.glob("*.xml")):
            img_file = img_dir / xml_file.name.replace(".xml", ".jpg")
            if img_file.exists():
                all_pairs.append((img_file, xml_file))
                count += 1

        print(f"{country}: {count} image-annotation pairs")

    print(f"\nTotal: {len(all_pairs)} pairs")

    if len(all_pairs) == 0:
        raise RuntimeError("No data found! Check the downloads above for errors.")

    # --- Shuffle and split: 85% train, 15% val ---
    random.seed(42)
    random.shuffle(all_pairs)
    split = int(0.85 * len(all_pairs))
    train_pairs = all_pairs[:split]
    val_pairs = all_pairs[split:]

    out_dir = Path("datasets/rdd2022_us_czech")

    for split_name, split_pairs in [("train", train_pairs), ("val", val_pairs)]:
        img_out = out_dir / "images" / split_name
        lbl_out = out_dir / "labels" / split_name
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)

        skipped = 0
        for img_path, xml_path in split_pairs:
            img = PILImage.open(img_path)
            w, h = img.size
            yolo_lines = convert_voc_to_yolo(xml_path, w, h)
            if not yolo_lines:
                skipped += 1
                continue
            shutil.copy(img_path, img_out / img_path.name)
            label_file = lbl_out / img_path.name.replace(".jpg", ".txt")
            label_file.write_text("\n".join(yolo_lines))

        total = len(list(img_out.glob("*.jpg")))
        print(f"{split_name}: {total} images ({skipped} skipped)")

    # --- Final sanity check ---
    train_count = len(list((out_dir / "images" / "train").glob("*.jpg")))
    val_count = len(list((out_dir / "images" / "val").glob("*.jpg")))
    if train_count < 1000 or val_count < 100:
        raise RuntimeError(
            f"Conversion produced too few usable images: "
            f"train={train_count}, val={val_count}. Cannot train."
        )

    print(f"\nDataset ready! train={train_count}, val={val_count}")

In [ ]:
_ps_setup()
if TRAINING_OPTION != "C":
    print(f"Skipping Option C training (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    import yaml
    from pathlib import Path

    print("=" * 60)
    print("  Starting Option C training (RDD2022 US+Czech v2.1)")
    print("=" * 60)
    print("  Expected duration:")
    print("    ~3-4 hours on T4 (free Colab) — 150 epochs at 1280px")
    print("    ~45-90 minutes on A100 (Colab Pro)")
    print()
    print("  Keep this tab open. Don't let your laptop sleep.")
    print("  Free Colab disconnects after ~90 min idle — wiggle the mouse now and then.")
    print("=" * 60)

    # Write dataset config
    dataset_config = {
        "path": str(Path("datasets/rdd2022_us_czech").resolve()),
        "train": "images/train",
        "val": "images/val",
        "names": {i: name for i, name in enumerate(CLASS_NAMES)},
    }

    config_path = Path("rdd2022_us_czech.yaml")
    config_path.write_text(yaml.dump(dataset_config, default_flow_style=False))
    print("Dataset config:")
    print(config_path.read_text())

    # Train YOLO11 on US+Czech RDD2022 with v2.1 settings
    model_rdd = YOLO(MODEL_SIZE_DET)

    results_rdd = model_rdd.train(
        data="rdd2022_us_czech.yaml",
        epochs=150,                       # v2.1: longer training
        imgsz=IMGSZ,
        batch=BATCH_SIZE,
        name="pavescan_rdd2022_uscz_v2",
        project="runs/detect",
        exist_ok=True,
        patience=50,                      # v2.1: more patience
        save=True,
        plots=True,
        optimizer="AdamW",
        seed=SEED,                        # E2: explicit ultralytics seed for reproducibility
        lr0=0.005,                        # E6: was default 0.01; caused EMA NaN/Inf at epoch 3
        warmup_epochs=5.0,                # E6: longer warmup for stability
        multi_scale=False,  # E6: was True; caused BatchNorm crash on small batch
        amp=False,                        # E7: was default True; EMA NaN/Inf at epoch 14+ (1280px+aggressive aug)
        # v2.1 aggressive augmentation (det — no copy_paste, no label_smoothing)
        hsv_h=0.02, hsv_s=0.8, hsv_v=0.5,
        fliplr=0.5, flipud=0.1,
        degrees=15.0,
        translate=0.15,
        scale=0.6,
        shear=2.0,
        perspective=0.0005,
        mosaic=1.0,
        mixup=0.15,
        # Focal loss focuses learning on rare classes (Pothole D40)
        fl_gamma=1.5,
    )

    print("\nOption C training complete!")
    print("Best weights: runs/detect/pavescan_rdd2022_uscz_v2/weights/best.pt")


In [ ]:
_ps_setup()
if TRAINING_OPTION != "C":
    print(f"Skipping Option C validation (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    from pathlib import Path

    BEST_PATH = "runs/detect/pavescan_rdd2022_uscz_v2/weights/best.pt"
    if not Path(BEST_PATH).exists():
        raise RuntimeError(
            f"Trained model not found at {BEST_PATH}. "
            "Did training succeed?"
        )

    best_model = YOLO(BEST_PATH)
    metrics = best_model.val()

    print("\n=== US + Czech RDD2022 v2 Performance ===")
    print(f"mAP50:    {metrics.box.map50:.3f}")
    print(f"mAP50-95: {metrics.box.map:.3f}")

    for i, name in enumerate(CLASS_NAMES):
        if i < len(metrics.box.ap50):
            print(f"  {name}: AP50 = {metrics.box.ap50[i]:.3f}")

In [ ]:
_ps_setup()
if TRAINING_OPTION != "C":
    print(f"Skipping Option C v1-vs-v2 comparison (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    import json
    from pathlib import Path

    metrics = {
        "version": "v2",
        "training_option": "C",
        "dataset": "RDD2022_US_Czech",
        "model_size": MODEL_SIZE_DET,
        "imgsz": IMGSZ,
        "batch_size": BATCH_SIZE,
        "epochs_planned": 150,
        "patience": 50,
        "multi_scale": False,
        "fl_gamma": 1.5,
        "class_names": CLASS_NAMES,
    }

    V2_PATH = "runs/detect/pavescan_rdd2022_uscz_v2/weights/best.pt"
    if not Path(V2_PATH).exists():
        raise RuntimeError(f"v2 weights not found at {V2_PATH}.")

    # --- v2 metrics on Option C val ---
    print("Evaluating v2 on RDD2022 (US+Czech) val set...")
    v2 = YOLO(V2_PATH)
    v2_val = v2.val(data="rdd2022_us_czech.yaml", imgsz=IMGSZ, batch=BATCH_SIZE, plots=False, verbose=False)
    metrics["v2_val_box_mAP50"]    = float(v2_val.box.map50)
    metrics["v2_val_box_mAP50_95"] = float(v2_val.box.map)
    metrics["v2_val_per_class_AP50"] = {
        CLASS_NAMES[i]: float(v2_val.box.ap50[i])
        for i in range(min(len(CLASS_NAMES), len(v2_val.box.ap50)))
    }

    # --- v1 baseline ---
    V1_PATH = "/content/pavescan_rdd2022.pt"
    if Path(V1_PATH).exists():
        print(f"\nFound v1 baseline at {V1_PATH}. Evaluating on the same val set...")
        try:
            v1 = YOLO(V1_PATH)
            v1_val = v1.val(data="rdd2022_us_czech.yaml", imgsz=IMGSZ, batch=BATCH_SIZE, plots=False, verbose=False)
            metrics["v1_val_box_mAP50"]    = float(v1_val.box.map50)
            metrics["v1_val_box_mAP50_95"] = float(v1_val.box.map)
        except Exception as e:
            print(f"v1 evaluation failed (non-fatal): {e}")
            metrics["v1_val_box_mAP50"]    = None
            metrics["v1_val_box_mAP50_95"] = None
    else:
        print(f"\nNo v1 baseline at {V1_PATH}. To enable v1-vs-v2 comparison:")
        print("  1. Click the folder icon in Colab's left sidebar")
        print("  2. Drag 'pavescan_rdd2022.pt' from your computer into /content/")
        print("  3. Re-run this single cell")
        metrics["v1_val_box_mAP50"]    = None
        metrics["v1_val_box_mAP50_95"] = None

    # --- Print comparison ---
    print("\n" + "=" * 60)
    print("  v1 vs v2 — RDD2022 (US + Czech)")
    print("=" * 60)
    if metrics["v1_val_box_mAP50"] is not None:
        d50 = metrics["v2_val_box_mAP50"] - metrics["v1_val_box_mAP50"]
        d95 = metrics["v2_val_box_mAP50_95"] - metrics["v1_val_box_mAP50_95"]
        print(f"  Box mAP50:    v1={metrics['v1_val_box_mAP50']:.3f}  ->  v2={metrics['v2_val_box_mAP50']:.3f}  (delta {d50:+.3f})")
        print(f"  Box mAP50-95: v1={metrics['v1_val_box_mAP50_95']:.3f}  ->  v2={metrics['v2_val_box_mAP50_95']:.3f}  (delta {d95:+.3f})")
    else:
        print(f"  Box mAP50:    v2={metrics['v2_val_box_mAP50']:.3f}    (v1 not evaluated)")
        print(f"  Box mAP50-95: v2={metrics['v2_val_box_mAP50_95']:.3f}  (v1 not evaluated)")
    print("\n  Per-class AP50:")
    for cls, ap in metrics["v2_val_per_class_AP50"].items():
        print(f"    {cls:20s}  {ap:.3f}")
    print("=" * 60)

    # --- Save metrics JSON ---
    metrics_path = Path("metrics_v2_det.json")
    metrics_path.write_text(json.dumps(metrics, indent=2))
    print(f"\nMetrics saved to: {metrics_path}")


In [ ]:
# Confidence calibration for Option C (RDD2022 US+Czech)
_ps_setup()
if TRAINING_OPTION != 'C':
    print(f"Skipping Option C calibration (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    from pathlib import Path
    import json

    V2_PATH = 'runs/detect/pavescan_rdd2022_uscz_v2/weights/best.pt'
    if not Path(V2_PATH).exists():
        raise RuntimeError(f'v2 weights not found at {V2_PATH}.')

    val_imgs_dir = Path('datasets/rdd2022_us_czech/images/val')
    val_lbls_dir = Path('datasets/rdd2022_us_czech/labels/val')
    if not val_imgs_dir.exists() or not val_lbls_dir.exists():
        if SKIP_CALIBRATION:
            print(f'SKIP_CALIBRATION=True; skipping calibration (would have used {val_imgs_dir})')
        else:
            raise RuntimeError(
                f'Calibration val data not found.\n'
                f'  Tried images: {val_imgs_dir}\n'
                f'  Tried labels: {val_lbls_dir}\n'
                f'If training succeeded, the val set should be on disk. '
                f'To skip calibration anyway, set SKIP_CALIBRATION = True in cell 7.'
            )
    else:
        print(f'Calibrating against {val_imgs_dir} ({sum(1 for _ in val_imgs_dir.glob("*.jpg"))} images)')
        v2_model = YOLO(V2_PATH)
        cal = calibrate_yolo_model(
            v2_model,
            val_imgs_dir=val_imgs_dir,
            val_lbls_dir=val_lbls_dir,
            imgsz=IMGSZ,
        )
        cal['version'] = 'v2'
        cal['training_option'] = 'C'
        cal['model'] = 'pavescan_rdd2022_uscz_v2'
        cal['model_size'] = MODEL_SIZE_DET
        cal['class_names'] = CLASS_NAMES

        out_path = Path('calibration_v2_det.json')
        out_path.write_text(json.dumps(cal, indent=2))
        print(f"\nCalibration saved to: {out_path}")
        print(f"  n_predictions = {cal['n_predictions']}, raw precision = {cal['precision_uncalibrated']:.3f}")
        print('  Drop this file in your repo at: pavescan-ai/models/calibration_v2_det.json')


In [ ]:
_ps_setup()
if TRAINING_OPTION != "C":
    print(f"Skipping Option C visualization (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    from IPython.display import Image, display
    from pathlib import Path

    run_dir = Path("runs/detect/pavescan_rdd2022_uscz_v2")

    if (run_dir / "results.png").exists():
        print("Training curves:")
        display(Image(filename=str(run_dir / "results.png"), width=800))

    if (run_dir / "val_batch0_pred.jpg").exists():
        print("\nSample predictions:")
        display(Image(filename=str(run_dir / "val_batch0_pred.jpg"), width=800))

    if (run_dir / "confusion_matrix.png").exists():
        print("\nConfusion matrix:")
        display(Image(filename=str(run_dir / "confusion_matrix.png"), width=600))

In [ ]:
_ps_setup()
if TRAINING_OPTION != "C":
    print(f"Skipping Option C download (TRAINING_OPTION = {TRAINING_OPTION!r})")
else:
    # _E7_GUARDED — wrap so the cell still works locally
    try:
        from google.colab import files
        _COLAB_OK = True
    except ImportError:
        _COLAB_OK = False
    from pathlib import Path
    import shutil

    BEST_PATH = "runs/detect/pavescan_rdd2022_uscz_v2/weights/best.pt"
    OUTPUT_NAME = "pavescan_rdd2022_v2.pt"

    if not Path(BEST_PATH).exists():
        raise RuntimeError(
            f"Trained weights not found at {BEST_PATH}. "
            "Cannot download — did training succeed?"
        )

    shutil.copy(BEST_PATH, OUTPUT_NAME)
    print(f"Downloading {OUTPUT_NAME} to your computer...")
    print(f"After download finishes, copy this file to: pavescan-ai/models/{OUTPUT_NAME}")
    if _COLAB_OK:
        files.download(OUTPUT_NAME)
    else:
        print("  (not in Colab — skipping download)")

    metrics_file = Path("metrics_v2_det.json")
    if metrics_file.exists():
        print(f"\nDownloading {metrics_file.name} (drop in your repo root)...")
        if _COLAB_OK:
            files.download(str(metrics_file))
        else:
            print("  (not in Colab — skipping download)")
